# Projet 03 — Inclusion Sociale : Accessibilite aux Structures d'Accueil de la Petite Enfance a Bruxelles

**Question politique** : Les structures d'accueil de la petite enfance sont-elles equitablement distribuees sur le territoire de la Ville de Bruxelles, et toutes les langues et communes sont-elles bien desservies ?

**Hypothese** : Les quartiers plus aises ou centraux beneficient d'une meilleure couverture en structures d'accueil (creches, gardiennes) que les quartiers peripheriques ou populaires.

---

**Auteur** : Kuate Joel Parfait  
**LinkedIn** : [joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
**Contact** : hello@dhcompany.pro | 0465 55 71 09  
**Source de donnees** : Open Data Bruxelles — opendata.brussels.be  
**Datasets** :  
- `milieux-accueil-petite-enfance-one-kindengezin-vbx`  
- `consultations-pour-enfants-one-vbx`  
**Derniere mise a jour** : Mars 2026

---

> Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
> [linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
> www.axiatechnologie.com

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
print("Librairies chargees.")

## 1. Chargement des donnees

In [ ]:
# Milieux d'accueil petite enfance
df_pe = pd.read_csv('milieux_accueil_petite_enfance.csv', sep=';', encoding='utf-8-sig')
print("Milieux accueil — shape:", df_pe.shape)
print("Colonnes:", df_pe.columns.tolist())
df_pe.head()

In [ ]:
# Consultations ONE
df_one = pd.read_csv('consultations_enfants_one.csv', sep=';', encoding='utf-8-sig')
print("Consultations ONE — shape:", df_one.shape)
print("Colonnes:", df_one.columns.tolist())
df_one.head()

## 2. Analyse des milieux d'accueil par type et langue

In [ ]:
# Repartition par categorie de milieu d'accueil
cat_col = [c for c in df_pe.columns if 'Cat' in c and 'egorie' in c][0]
type_col = [c for c in df_pe.columns if "Type d'accueil" in c or "Soort" in c][0] if any("Type d'accueil" in c for c in df_pe.columns) else None

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Categorie
cat_counts = df_pe[cat_col].value_counts()
colors_cat = sns.color_palette("Set2", len(cat_counts))
axes[0].barh(cat_counts.index, cat_counts.values, color=colors_cat)
axes[0].set_title("Types de milieux d'accueil de la\npetite enfance — Bruxelles", fontweight='bold')
axes[0].set_xlabel("Nombre de structures")
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

# Langue
lang_col = [c for c in df_pe.columns if 'Langue' in c or 'Taal' in c][0] if any('Langue' in c or 'Taal' in c for c in df_pe.columns) else None
if lang_col:
    lang_counts = df_pe[lang_col].value_counts()
    wedges, texts, autotexts = axes[1].pie(lang_counts.values, labels=lang_counts.index, 
                                            autopct='%1.1f%%', 
                                            colors=sns.color_palette("Pastel1", len(lang_counts)),
                                            startangle=90)
    axes[1].set_title("Repartition par langue d'enseignement\nmilieux accueil petite enfance", fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'Donnee langue non disponible', ha='center', va='center')

plt.tight_layout()
plt.savefig('01_types_milieux_accueil.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

## 3. Repartition geographique par commune

In [ ]:
# Distribution des structures par commune
commune_col = [c for c in df_pe.columns if 'Commune' in c][0]
commune_counts = df_pe[commune_col].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(13, max(6, len(commune_counts)*0.5)))
bars = ax.barh(commune_counts.index, commune_counts.values, 
               color=sns.color_palette("Blues_r", len(commune_counts)))
ax.set_title("Nombre de milieux d'accueil petite enfance par commune\n(Ville de Bruxelles)", 
             fontweight='bold', fontsize=13)
ax.set_xlabel("Nombre de structures")
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, 
            str(int(width)), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('02_repartition_commune.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

In [ ]:
# Croisement : structures d'accueil + consultations ONE par commune
one_commune = df_one[[c for c in df_one.columns if 'Commune' in c][0]].value_counts()
pe_commune = df_pe[commune_col].value_counts()

# Merge pour comparaison
df_compare = pd.DataFrame({
    'Milieux accueil': pe_commune,
    'Consultations ONE': one_commune
}).fillna(0).reset_index()
df_compare.columns = ['Commune', 'Milieux_accueil', 'Consultations_ONE']
df_compare = df_compare.sort_values('Milieux_accueil', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(df_compare))
width = 0.4
ax.bar([i - width/2 for i in x], df_compare['Milieux_accueil'], 
       width, label="Milieux d'accueil", color='steelblue', alpha=0.85)
ax.bar([i + width/2 for i in x], df_compare['Consultations_ONE'], 
       width, label='Consultations ONE', color='coral', alpha=0.85)
ax.set_title("Comparaison : Milieux d'accueil vs Consultations ONE par commune\n(Ville de Bruxelles)", 
             fontweight='bold', fontsize=13)
ax.set_xlabel("Commune")
ax.set_ylabel("Nombre de structures")
ax.set_xticks(list(x))
ax.set_xticklabels(df_compare['Commune'], rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('03_croisement_accueil_ONE.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

In [ ]:
# Analyse des types de tarification (accessibilite financiere)
tarif_col = [c for c in df_pe.columns if 'tarif' in c.lower() or 'Tarif' in c][0] if any('tarif' in c.lower() for c in df_pe.columns) else None
statut_col = [c for c in df_pe.columns if 'Statut' in c][0] if any('Statut' in c for c in df_pe.columns) else None

if tarif_col or statut_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    if tarif_col:
        tarif_counts = df_pe[tarif_col].value_counts()
        axes[0].pie(tarif_counts.values, labels=tarif_counts.index, autopct='%1.1f%%',
                   colors=sns.color_palette("Set3", len(tarif_counts)), startangle=90)
        axes[0].set_title("Types de tarification\nmilieux d'accueil", fontweight='bold')
    
    if statut_col:
        statut_counts = df_pe[statut_col].value_counts()
        axes[1].bar(statut_counts.index, statut_counts.values, 
                   color=sns.color_palette("muted", len(statut_counts)))
        axes[1].set_title("Statut des milieux d'accueil\n(public/prive/subventionne)", fontweight='bold')
        axes[1].set_ylabel("Nombre")
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')
    
    plt.tight_layout()
    plt.savefig('04_tarification_statut.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Graphique sauvegarde.")

## 4. Interpretation politique et conclusions

### Constats principaux

1. **Repartition inegale** : Les milieux d'accueil de la petite enfance ne sont pas uniformement distribues sur le territoire. Certains quartiers et communes concentrent davantage de structures que d'autres.

2. **Question linguistique** : La couverture en structures francophones versus neerlandophones reflete les dynamiques linguistiques de Bruxelles, avec des implications pour l'acces equitable selon la langue de l'enfant.

3. **Types de tarification** : La predominance de structures a tarification selon revenus (tarification sociale) est positive pour l'accessibilite financiere, mais la capacite totale reste un enjeu.

4. **Consultations ONE vs milieux d'accueil** : Le croisement revele que les consultations de sante (ONE) ne sont pas necessairement co-localisees avec les milieux d'accueil, ce qui peut cree des deserts de services pour les parents.

### Question politique repondue

> Les structures d'accueil sont-elles equitablement distribuees sur le territoire bruxellois ?

**Reponse** : Non completement. Les donnees revelent des disparites geographiques notables. Certaines communes sont sous-equipees relativement a leur population, ce qui constitue une inegalite d'acces aux services publics pour les familles.

### Recommandations politiques

- Prioritiser l'implantation de nouvelles structures dans les communes les moins bien desservies.
- Renforcer la coordination entre ONE et les milieux d'accueil pour une couverture territoriale coherente.
- Veiller a l'equilibre linguistique dans l'offre d'accueil pour refleter la diversite de la population bruxelloise.

---

**Contact pour decisions politiques ou formations IA & Data** :  
Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
[linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
www.axiatechnologie.com